In [2]:
import janitor
import matplotlib.pyplot as plt
import missingno
import nhanes.load
import numpy as np
import pandas as pd
import scipy.stats
import seaborn as sns
import session_info
import sklearn.compose
import sklearn.impute
import sklearn.preprocessing
import statsmodels.api as sm
import statsmodels.datasets
import statsmodels.formula.api as smf

from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import BayesianRidge, Ridge
from sklearn.neighbors import KNeighborsRegressor
from statsmodels.graphics.mosaicplot import mosaic
from pandas_missing_extension import MissingMethods

In [3]:
%matplotlib inline

sns.set(
    rc={
        "figure.figsize": (8, 6)
    }
)

sns.set_style("whitegrid")

In [4]:
airquality_df = (
    sm.datasets.get_rdataset("airquality")
    .data
    .clean_names(
        case_type = "snake"
    )
    .add_column("year", 1973)
    .assign(
        date = lambda df: pd.to_datetime(df[["year", "month", "day"]])
    )
    .sort_values(by = "date")
    .set_index("date")
)

airquality_df.head(2)

c:\Users\SEBASTIAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas_flavor\register.py:157: FutureWarning: This function will be deprecated in a 1.x release. Please use `pd.DataFrame.assign` instead.
  return method(self._obj, *args, **kwargs)


,ozone,solar_r,wind,temp,month,day,year
date,,,,,,,
1973-05-01,41.0,190.0,7.4,67,5,1,1973
1973-05-02,36.0,118.0,8.0,72,5,2,1973


In [ ]:
#No se pueden comparar estos modelos porque no tienen la misma cantidad de datos. En el primer modelo hay 116, en el segundo 111

smf.ols(formula= "temp ~ ozone", data= airquality_df).fit().summary().tables[0]
smf.ols(formula= "temp ~ ozone + solar_r", data= airquality_df).fit().summary().tables[0]

In [ ]:
#CARGAR Y PROCESAR LOS DATOS

#Cargar los datos
nhanes_raw_df= nhanes.load.load_NHANES_data(year="2017-2018").clean_names(case_type="snake") #clean_names(case_type="snake") es para que separe los nombres de las variables, ej: GeneralHealthCondition -> general_health_condition

#Procesar los datos 

# Voy a seleccionar unas columnas y les voy a cambiar el nombre. Luego voy a decir que datos quiero que lea como faltantes.
#Luego las voy a organizar en orden de las que mas tienen datos faltantes. Luego, voy a eliminar todos los datos faltantes de la columna "diabetes".
#Por ultimo, voy a convertir la columna "diabetes" en entero, ya que en el programa la lee como un objeto.

nhanes_df = (
    nhanes_raw_df.select_columns(
        "general_health_condition",
        "age_in_years_at_screening",
        "gender",
        "current_selfreported_height_inches",
        "current_selfreported_weight_pounds",
        "doctor_told_you_have_diabetes",
        "60_sec_pulse30_sec_pulse2",
        "total_cholesterol_mgdl"
    )
    .rename_columns(
        {
            "age_in_years_at_screening": "age",
            "current_selfreported_height_inches": "height",
            "current_selfreported_weight_pounds": "weight",
            "doctor_told_you_have_diabetes": "diabetes",
            "60_sec_pulse30_sec_pulse2": "pulse",
            "total_cholesterol_mgdl": "total_cholesterol"
        }
    )
    .replace(
        {
            "height": {
                9999: np.nan,
                7777: np.nan
            },
            "weight": {
                9999: np.nan,
                7777: np.nan
            },
            "diabetes": {
                "Borderline": np.nan
            }
        }
    )
    .missing.sort_variables_by_missingness()
    .dropna(
        subset = ["diabetes"],
        how = "any"
    )
    .transform_column(
        column_name = "diabetes",
        function = lambda s: s.astype(int),
        elementwise = False
    )
)

nhanes_df.shape

In [4]:
#VISUALIZAR Y ELIMINAR DATOS FALTANTES

#nhanes_df.missing.sort_variables_by_missingness().pipe(missingno.matrix, sort= "descending") #Graficar en orden a los que mas tienen datos faltantes
#nhanes_df.missing.sort_variables_by_missingness().missing.missing_upsetplot() #Observar las relaciones entre las columnas que tienen datos faltantes

#Eliminar las filas que contengan datos faltantes en todas las 5 columnas. 
nhanes_df= nhanes_df.dropna(
    subset= ["pulse", "total_cholesterol", "general_health_condition", "weight", "height"], how= "all"
).missing.sort_variables_by_missingness()

In [16]:
#CONOCER QUE TIPO DE DATO ES (MCAR, MAR O MNAR) A TRAVES DE UNA T-TEST

# Estos son los metodos que se usan en "alternative" para la prueba t-student
# two-sided: las medias de las distribuciones subyacentes a las muestras son desiguales.
# less: la media de la distribución subyacente a la primera muestra es menor que la media de la distribución subyacente a la segunda muestra.
# greater: la media de la distribución subyacente a la primera muestra es mayor que la media de la distribución subyacente a la segunda muestra.

# seleccionar las columnas 'gender' y 'weight' y asigna el resultado a la variable female_weight y male_weight
female_weight, male_weight = (
    nhanes_df
    .select_columns("gender", "weight") 
    .transform_column("weight", lambda x: x.isna(), elementwise = False) # Transforma la columna 'weight' usando una función 
    # que verifica si los valores son NaN, y asigna el resultado a la variable weight
    .groupby("gender")
    .weight
    .pipe(
        lambda df: (
            df.get_group("Female"),
            df.get_group("Male")
        ) # Aplica una función que devuelve los grupos correspondientes
    # a 'Female' y 'Male' y asigna el resultado a las variables female_weight y male_weight
    )
)

scipy.stats.ttest_ind(a = female_weight, b = male_weight, alternative="two-sided") #Realizar la prueba t-student para los dos grupos

#En este caso, el valor p fue mayor a 0.05, asi que signfica que las medias no tienen diferencias significativas

c:\Users\SEBASTIAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas_flavor\register.py:157: FutureWarning: This function will be deprecated in a 1.x release. Please use `jn.select` instead.
  return method(self._obj, *args, **kwargs)


TtestResult(statistic=-0.3621032192538131, pvalue=0.7172855918077239, df=7155.0)

In [ ]:
#VISUALIZAR LOS DATOS FALTANTES A TRAVES DE UN MOSAICO

fig, ax = plt.subplots(figsize=(10, 10))

(
    nhanes_df.select_columns("weight", "general_health_condition", "gender")
    .assign(weight=lambda df: df.weight.isna().replace([True, False], ["NA", "!NA"]))
    .groupby(
        ["gender", "general_health_condition", "weight"], dropna=False, as_index=True
    )
    .size()
    .pipe(
        lambda df: mosaic(
            data=df,
            properties=lambda key: {"color": "r" if "NA" in key else "gray"},
            ax=ax,
            horizontal=True,
            axes_label=True,
            title="",
            labelizer=lambda key: "",
        )
    )
)

ax.grid(False)

In [18]:
#TRATAMIENTO DE VALORES CATEGORICAS

nhanes_transform_df= nhanes_df.copy(deep=True) #Hacer una copia del conjunto de datos

#Convertir datos categoricos en numericos
encoder= sklearn.preprocessing.OrdinalEncoder()
X= [["Male"],["Female"],["Female"]]
encoder.fit_transform(X) 

#Encontrar la inversa. Esto significa que pueda ingresar un numero y sepa a que variable categorica corresponde
encoder.inverse_transform([[1],[0],[0]])

array([['Male'],
       ['Female'],
       ['Female']], dtype=object)

In [ ]:
categorical_columns= nhanes_df.select_dtypes(include=[object, "category"]).columns #Seleccionar solo variables categoricas

#Transformar de datos categoricas a datos numericos
categorical_transformer= sklearn.compose.make_column_transformer(
    (sklearn.preprocessing.OrdinalEncoder(), categorical_columns),
    remainder= "passthrough"
) 

pd.DataFrame(
    categorical_transformer.fit_transform(nhanes_df),
    columns= categorical_transformer.get_feature_names_out()
)

# Transforma las características categóricas de nhanes_df utilizando categorical_transformer y crea un DataFrame
nhanes_transformed_df = (
    pd.DataFrame(
        categorical_transformer.fit_transform(nhanes_df),
        columns = categorical_transformer.get_feature_names_out(),
        index = nhanes_df.index
    )
    # Renombra las columnas para eliminar el prefijo agregado por OrdinalEncoder
    .rename_columns(
        function = lambda x: x.removeprefix("ordinalencoder__")
    )
    # Renombra las columnas para eliminar el prefijo agregado por OneHotEncoder
    .rename_columns(
        function = lambda x: x.removeprefix("onehotencoder__")
    )
    # Renombra las columnas para eliminar el prefijo agregado por remainder del ColumnTransformer
    .rename_columns(
        function = lambda x: x.removeprefix("remainder__")
    )
)

nhanes_transformed_df

In [ ]:
# ONE HOT ENCODING
#Consiste en crear una matriz de unos y ceros donde el 1 representa la presencia de una caracteristica o categoria de la 
#variable de interes y 0 la ausencia de la misma

nhanes_transformed_df2= nhanes_df.copy(deep= True) #Realizar una copia del Dataframe

#Usando pd.get_dummies
nhanes_transformed_df2.select_columns("general_health_condition").pipe(pd.get_dummies) #Usando pd.get_dummies

#Con Sklearn

# Definir la transformación de columnas
transformer = sklearn.compose.make_column_transformer(
    (sklearn.preprocessing.OrdinalEncoder(), ["gender"]), # Codifica la columna "gender" como valores numéricos
    (sklearn.preprocessing.OneHotEncoder(), ["general_health_condition"]), # Aplica one-hot encoding a la columna "general_health_condition"
    remainder="passthrough" # Mantiene las columnas no especificadas sin modificar
)

# Aplicar la transformación al DataFrame nhanes_df
nhanes_transformed_df2 = (
    pd.DataFrame(
        transformer.fit_transform(nhanes_df), # Aplica la transformación al DataFrame nhanes_df
        columns = transformer.get_feature_names_out(), # Asigna nombres a las columnas transformadas
        index = nhanes_df.index # Mantiene los índices del DataFrame original
    )
    .rename_columns(
        function = lambda x: x.removeprefix("ordinalencoder__") # Elimina el prefijo "ordinalencoder__" de los nombres de las columnas
    )
    .rename_columns(
        function = lambda x: x.removeprefix("onehotencoder__") # Elimina el prefijo "onehotencoder__" de los nombres de las columnas
    )
    .rename_columns(
        function = lambda x: x.removeprefix("remainder__") # Elimina el prefijo "remainder__" de los nombres de las columnas
    )
)

nhanes_transformed_df

In [ ]:
#IMPUTACION POR MEDIAS
nhanes_df.transform_column("height", lambda x: x.fillna(x.mean()), elementwise= False) #Usando Janitor

#IMPUTACION POR LLENADO HACIA ATRAS E IMPUTACION POR LLENADO HACIA ADELANTE
nhanes_df.select_columns("height", "weight").fillna(method= "ffill") # Rellenar hacia adelante
nhanes_df.select_columns("height", "weight").fillna(method= "bfill") # Rellenar hacia atras

#Imputación dentro de dominios e imputación a través de variables correlacionadas
(
    nhanes_df
    .select_columns("height", "weight", "gender", "diabetes", "general_health_condition")
    .sort_values(
        by = ["gender", "diabetes", "general_health_condition", "height"],
        ascending = True
    )
    .transform_column(
        "weight",
        lambda x: x.ffill(),
        elementwise = False
    )
)

#IMPUTACION POR INTERPOLACION
plt.figure(figsize=(20,10))
airquality_df.select_columns("ozone").pipe(
    lambda df: (
        df.ozone.interpolate(method= "linear").plot(color= "red", marker= "o", alpha= 6/9, linestyle= "dashed"),
        df.ozone.plot(color= "#313638", marker= "o")
    )
)

plt.figure(figsize=(20,10))
airquality_df.select_columns("ozone").pipe(
    lambda df: (
        df.ozone.interpolate(method= "quadratic").plot(color= "red", marker= "o", alpha= 6/9, linestyle= "dashed"),
        df.ozone.plot(color= "#313638", marker= "o")
    )
)

plt.figure(figsize=(20,10))
airquality_df.select_columns("ozone").pipe(
    lambda df: (
        df.ozone.interpolate(method= "nearest").plot(color= "red", marker= "o", alpha= 6/9, linestyle= "dashed"),
        df.ozone.plot(color= "#313638", marker= "o")
    )
)

#En este caso, el mejor modelo, resulto siendo el lineal, entonces para guardarlo

airquality_df["ozone"] = airquality_df.ozone.interpolate(method="linear")
airquality_df

In [ ]:
#IMPUTACION POR KNN (K-VECINOS)
nhanes_transformed_df

knn_imputer = sklearn.impute.KNNImputer()
nhanes_df_knn = nhanes_transformed_df.copy(deep=True)

nhanes_df_knn.iloc[:,:]= knn_imputer.fit_transform(nhanes_transformed_df).round()

#Visualizar
(
    pd.concat(
        [
            nhanes_df_knn,
            nhanes_df.missing.create_shadow_matrix(True, False, suffix="_imp", only_missing=True)
        ],
        axis=1
    )
    .missing.scatter_imputation_plot(x="height", y="weight")
)

In [20]:
#Ordenamiento por cantidad de valores faltantes
nhanes_transformed_df

knn_imputer = sklearn.impute.KNNImputer()
nhanes_df_knn = nhanes_transformed_df.missing.sort_variables_by_missingness(ascending= True).copy(deep=True)

nhanes_df_knn.iloc[:,:]= knn_imputer.fit_transform(nhanes_transformed_df.missing.sort_variables_by_missingness(ascending= True)).round()

In [ ]:
#IMPUTACION BASADA EN MODELOS

#Basada en regresion lineal
nhanes_model_df = (
    nhanes_df
    .select_columns("height", "weight", "gender", "age")
    .sort_values(by = "height") #Se debe organizar por un valor
    .transform_column(
        "weight", #Se selecciona cualquiera para que el modelo inicie. Cuando haya terminado el procedimiento, se cambia y se pone otra variable a imputar   
        lambda x: x.ffill(),
        elementwise = False
    ))

height_ols = (
    nhanes_model_df
    .pipe(
        lambda df: smf.ols("height ~ weight + gender + age",  data=df) #Crear el modelo
    )
    .fit()
)

ols_imputed_values = (
    nhanes_model_df
    .pipe(
        lambda df: df[df.height.isna()]
    )
    .pipe(
        lambda df: height_ols.predict(df).round()
    )
)

nhanes_model_df.loc[nhanes_model_df.height.isna(), ["height"]] = ols_imputed_values 


In [30]:
#IMPUTACIONES MULTIPLES POR ECUACIONES ENCADENADAS (MICE)

mice_imputer = sklearn.impute.IterativeImputer(estimator=BayesianRidge()) #Imputar usando el estimador Bayesian Ridge 
nhanes_mice_df = nhanes_transformed_df.copy(deep = True) 
nhanes_mice_df.iloc[:, :] = mice_imputer.fit_transform(nhanes_mice_df).round() #Aplicar la imputacion y redondear los valores a enteros

In [32]:
#TRANSFORMACION INVERSA DE LOS DATOS
#Este metodo es para transformar los valores que son categorias, finalmente en categorias

nhanes_imputated_df = nhanes_mice_df.copy(deep=True)

nhanes_imputated_df[categorical_columns] = (
    categorical_transformer # funcion que transforma los valores
    .named_transformers_ # nombre de las transformaciones realizadas a los datos
    .ordinalencoder # transformador específico para variables categóricas ordinales
    .inverse_transform( #se aplica la transformación inversa a las variables categóricas imputadas
        X = nhanes_mice_df[categorical_columns] # realizar la transformación inversa a las columnas categóricas
    )
)

nhanes_df.general_health_condition.value_counts() #Comparar el antes de la imputacion
nhanes_imputated_df.general_health_condition.value_counts()  #Comparar el despues de la imputacion

nhanes_mice_df.missing.number_missing() #Comprobar si hay datos faltantes

0